
# 02 — Baseline Models, Duration Diagnostic and Hyperparameter Tuning

This notebook is the executable quantitative source of truth for the baseline/tuning comparison.

### Correction applied

An earlier version could skip tuning when cached candidate CSV/JSON files already existed (`FORCE_RETUNE=False`). That allowed stale selected parameters to survive into later outputs. This version removes cache-dependent winner selection entirely: **all four searches are recomputed from the dataset every time the tuning cell is run**.

The analysis separates two questions:

1. **Duration effect under controlled baselines:** within each model family, pre-call and +`duration` use identical untuned settings, so only the predictor set changes.
2. **Duration effect after tuning:** LR pre-call, LR +`duration`, HGB pre-call and HGB +`duration` are tuned separately. Each winner is selected by PR-AUC first and ROC-AUC only as a tie-breaker.


In [ ]:

from pathlib import Path
from inspect import signature
import json, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, ParameterGrid, ParameterSampler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss
)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)
SEED=42

DATA_CANDIDATES=[Path('term-deposit-marketing-2020.csv'), Path('data/term-deposit-marketing-2020.csv')]
DATA=next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA is None:
    raise FileNotFoundError('Place term-deposit-marketing-2020.csv beside this notebook or in data/.')

df=pd.read_csv(DATA)
df['y_binary']=df['y'].eq('yes').astype(int)
y=df['y_binary'].to_numpy(); idx=np.arange(len(df))
NUM=['age','balance','day','campaign']
NUMD=['age','balance','day','duration','campaign']
CAT=['job','marital','education','default','housing','loan','contact','month']
PRE=NUM+CAT; ALL=NUMD+CAT

assert len(df)==40000
assert int(y.sum())==2896
assert abs(y.mean()-0.0724)<1e-12
print(f'Dataset: {len(df):,} rows | subscribers: {int(y.sum()):,} | rate: {y.mean():.2%}')


## 1. Reproducible split

In [ ]:

dev_idx,test_idx=train_test_split(idx,test_size=.20,stratify=y,random_state=SEED)
train_idx,val_idx=train_test_split(dev_idx,test_size=.25,stratify=y[dev_idx],random_state=SEED)
hgb_tune_idx,_=train_test_split(train_idx,train_size=10000,stratify=y[train_idx],random_state=SEED)

split_summary=pd.DataFrame({
    'split':['LR tuning train','tuning validation','HGB tuning subsample','development total','final untouched test'],
    'n':[len(train_idx),len(val_idx),len(hgb_tune_idx),len(dev_idx),len(test_idx)],
    'base_rate':[y[train_idx].mean(),y[val_idx].mean(),y[hgb_tune_idx].mean(),y[dev_idx].mean(),y[test_idx].mean()]
})
display(split_summary.round(4))


## 2. Controlled baseline definitions and search spaces

In [ ]:

def preprocessor(features):
    return ColumnTransformer([
        ('numeric',StandardScaler(),[f for f in features if f in NUMD]),
        ('categorical',OneHotEncoder(handle_unknown='ignore',sparse_output=False),[f for f in features if f in CAT])
    ])

def lr_model(params):
    q=params.copy(); penalty=q.pop('penalty')
    if signature(LogisticRegression).parameters['penalty'].default=='deprecated':
        q['l1_ratio']=1.0 if penalty=='l1' else 0.0
    else:
        q['penalty']=penalty
    return LogisticRegression(solver='liblinear',max_iter=3000,random_state=SEED,**q)

def hgb_model(params):
    return HistGradientBoostingClassifier(random_state=SEED,**params)

def baseline_model(name):
    if name=='LR':
        return LogisticRegression(C=1.0,max_iter=100,tol=1e-4,random_state=SEED)
    return HistGradientBoostingClassifier(
        loss='log_loss',learning_rate=.1,max_iter=100,max_leaf_nodes=31,max_depth=None,
        min_samples_leaf=20,l2_regularization=0,max_bins=255,early_stopping='auto',
        class_weight=None,random_state=SEED)

LR_GRID=list(ParameterGrid({'C':np.logspace(-3,2,15),'penalty':['l1','l2'],'class_weight':[None,'balanced']}))
HGB_SPACE={
    'max_iter':[50,100,150,200,300,400], 'max_depth':[3,4,5,6,8,None],
    'learning_rate':[.01,.03,.05,.1,.2], 'max_leaf_nodes':[15,31,63,127],
    'l2_regularization':[0,.1,.5,1,2], 'min_samples_leaf':[10,20,30,50],
    'class_weight':[None,'balanced']}
HGB_CANDIDATES=list(ParameterSampler(HGB_SPACE,n_iter=100,random_state=SEED))

print('LR candidates per regime:',len(LR_GRID))
print('HGB full search space:',int(np.prod([len(v) for v in HGB_SPACE.values()])))
print('HGB sampled candidates per regime:',len(HGB_CANDIDATES))
print('Total candidate configurations:',2*len(LR_GRID)+2*len(HGB_CANDIDATES))


## 3. Four fresh searches — no cached winner selection

In [ ]:

def tune_lr(features):
    pp=preprocessor(features)
    Xt=pp.fit_transform(df.loc[train_idx,features]); Xv=pp.transform(df.loc[val_idx,features])
    rows=[]
    for run,p in enumerate(LR_GRID,1):
        m=lr_model(p); m.fit(Xt,y[train_idx]); s=m.predict_proba(Xv)[:,1]
        rows.append({'run':run,'validation_pr_auc':average_precision_score(y[val_idx],s),
                     'validation_roc_auc':roc_auc_score(y[val_idx],s),'params':json.dumps(p,sort_keys=True)})
    z=pd.DataFrame(rows).sort_values(['validation_pr_auc','validation_roc_auc'],ascending=False).reset_index(drop=True)
    return json.loads(z.iloc[0]['params']),z

def tune_hgb(features):
    pp=preprocessor(features)
    Xt=pp.fit_transform(df.loc[hgb_tune_idx,features]); Xv=pp.transform(df.loc[val_idx,features])
    rows=[]
    for run,p in enumerate(HGB_CANDIDATES,1):
        q={**p,'early_stopping':True}
        m=hgb_model(q); m.fit(Xt,y[hgb_tune_idx]); s=m.predict_proba(Xv)[:,1]
        rows.append({'run':run,'validation_pr_auc':average_precision_score(y[val_idx],s),
                     'validation_roc_auc':roc_auc_score(y[val_idx],s),'params':json.dumps(q,sort_keys=True)})
    z=pd.DataFrame(rows).sort_values(['validation_pr_auc','validation_roc_auc'],ascending=False).reset_index(drop=True)
    return json.loads(z.iloc[0]['params']),z

selected={}; searches={}
for key,kind,features in [
    ('LR_pre_call','LR',PRE),('LR_with_duration','LR',ALL),
    ('HGB_pre_call','HGB',PRE),('HGB_with_duration','HGB',ALL)]:
    selected[key],searches[key]=(tune_lr(features) if kind=='LR' else tune_hgb(features))

# Reproducibility / stale-cache guard: each selected parameter set must be row 1 of its OWN search table.
for key in selected:
    rank1=json.loads(searches[key].iloc[0]['params'])
    assert selected[key]==rank1, f'{key}: selected parameters do not match rank-1 candidate'

search_summary=pd.DataFrame([{
    'configuration':key,
    'winning_run':int(searches[key].iloc[0]['run']),
    'validation_pr_auc':searches[key].iloc[0]['validation_pr_auc'],
    'validation_roc_auc':searches[key].iloc[0]['validation_roc_auc'],
    'selected_parameters':json.dumps(selected[key],sort_keys=True)} for key in selected])
display(search_summary.round({'validation_pr_auc':6,'validation_roc_auc':6}))



### Verified rerun findings from the supplied 40,000-row dataset

| Configuration | Winning run | Validation PR-AUC | Validation ROC-AUC | Selected hyperparameters |
|---|---:|---:|---:|---|
| LR pre-call | **46** | **0.205127** | **0.671547** | L2; `C=8.483428982440726`; `class_weight=None`; solver=`liblinear` |
| LR + duration | **21** | **0.560634** | **0.940139** | L1; `C=0.0610540229658533`; `class_weight=None`; solver=`liblinear` |
| HGB pre-call | **79** | **0.222145** | **0.695497** | `max_iter=200`, `max_depth=8`, `learning_rate=0.05`, `max_leaf_nodes=15`, `min_samples_leaf=20`, `l2_regularization=0`, `class_weight=None`, `early_stopping=True` |
| HGB + duration | **72** | **0.560812** | **0.944407** | `max_iter=200`, `max_depth=3`, `learning_rate=0.05`, `max_leaf_nodes=127`, `min_samples_leaf=10`, `l2_regularization=2`, `class_weight='balanced'`, `early_stopping=True` |

These results replace the stale cached selections that had previously reported LR +duration as L2 / `C=0.138949...` and had incorrectly reused HGB run 79 for +duration.


In [ ]:

for key in ['LR_pre_call','LR_with_duration','HGB_pre_call','HGB_with_duration']:
    print(f'\n{key} — top 5 candidates')
    display(searches[key].head(5).round({'validation_pr_auc':6,'validation_roc_auc':6}))


## 4. Audit all tuned and untuned models on the untouched holdout

In [ ]:

def make_pipeline(features,model):
    return Pipeline([('preprocess',preprocessor(features)),('model',model)])

def evaluate_holdout(configuration,model_name,stage,features,model):
    pipe=make_pipeline(features,model)
    pipe.fit(df.loc[dev_idx,features],y[dev_idx])
    score=pipe.predict_proba(df.loc[test_idx,features])[:,1]
    pred=(score>=.5).astype(int)
    return {
        'configuration':configuration,'model':model_name,'stage':stage,
        'predictors':'Pre-call' if features==PRE else '+ duration',
        'accuracy':accuracy_score(y[test_idx],pred),
        'precision':precision_score(y[test_idx],pred,zero_division=0),
        'recall':recall_score(y[test_idx],pred,zero_division=0),
        'f1':f1_score(y[test_idx],pred,zero_division=0),
        'roc_auc':roc_auc_score(y[test_idx],score),
        'pr_auc':average_precision_score(y[test_idx],score),
        'brier':brier_score_loss(y[test_idx],score)}

rows=[]
for config,name,stage,features,model in [
    ('LR untuned pre-call','LR','Untuned',PRE,baseline_model('LR')),
    ('LR untuned + duration','LR','Untuned',ALL,baseline_model('LR')),
    ('LR tuned pre-call','LR','Tuned',PRE,lr_model(selected['LR_pre_call'])),
    ('LR tuned + duration','LR','Tuned',ALL,lr_model(selected['LR_with_duration'])),
    ('HGB untuned pre-call','HGB','Untuned',PRE,baseline_model('HGB')),
    ('HGB untuned + duration','HGB','Untuned',ALL,baseline_model('HGB')),
    ('HGB tuned pre-call','HGB','Tuned',PRE,hgb_model(selected['HGB_pre_call'])),
    ('HGB tuned + duration','HGB','Tuned',ALL,hgb_model(selected['HGB_with_duration']))]:
    rows.append(evaluate_holdout(config,name,stage,features,model))

holdout=pd.DataFrame(rows)
display(holdout[['model','stage','predictors','accuracy','precision','recall','f1','roc_auc','pr_auc','brier']].round(4))



### Verified audited performance

| Model | Stage | Predictors | Accuracy | Precision | Recall | F1 | ROC-AUC | PR-AUC | Brier |
|---|---|---|---:|---:|---:|---:|---:|---:|---:|
| LR | Untuned | Pre-call | 0.9283 | 0.5758 | 0.0328 | 0.0621 | 0.7112 | 0.2417 | 0.0621 |
| LR | Untuned | + duration | 0.9381 | 0.6556 | 0.3057 | 0.4170 | 0.9324 | 0.5250 | 0.0467 |
| LR | Tuned | Pre-call | 0.9281 | 0.5588 | 0.0328 | 0.0620 | 0.7097 | 0.2403 | 0.0622 |
| LR | Tuned | + duration | 0.9365 | 0.6461 | 0.2712 | 0.3820 | 0.9332 | 0.5207 | 0.0471 |
| HGB | Untuned | Pre-call | 0.9280 | 0.5349 | 0.0397 | 0.0740 | 0.7341 | 0.2683 | 0.0600 |
| HGB | Untuned | + duration | 0.9411 | 0.6399 | 0.4266 | 0.5119 | 0.9515 | 0.5852 | 0.0411 |
| HGB | Tuned | Pre-call | 0.9289 | 0.6190 | 0.0449 | 0.0837 | 0.7339 | 0.2728 | 0.0598 |
| HGB | Tuned | + duration | 0.8660 | 0.3386 | 0.8929 | 0.4910 | 0.9431 | 0.5498 | 0.0979 |


## 5. Duration effect and tuning effect

In [ ]:

h=holdout.set_index('configuration')
rows=[]
for model in ['LR','HGB']:
    up=h.loc[f'{model} untuned pre-call']; ud=h.loc[f'{model} untuned + duration']
    tp=h.loc[f'{model} tuned pre-call']; td=h.loc[f'{model} tuned + duration']
    for comparison,a,b in [
        ('Untuned: +duration minus pre-call',up,ud),
        ('Tuned: +duration minus pre-call',tp,td),
        ('Pre-call: tuned minus untuned',up,tp),
        ('+duration: tuned minus untuned',ud,td)]:
        rows.append({'model':model,'comparison':comparison,
                     'delta_roc_auc':b.roc_auc-a.roc_auc,'delta_pr_auc':b.pr_auc-a.pr_auc,
                     'delta_f1':b.f1-a.f1,'delta_precision':b.precision-a.precision,
                     'delta_recall':b.recall-a.recall,'delta_accuracy':b.accuracy-a.accuracy})
delta_table=pd.DataFrame(rows)
display(delta_table.round(4))



### Interpretation of duration and tuning

The **untuned baseline-to-baseline comparison** isolates the addition of `duration`: LR gains about **+0.221 ROC-AUC and +0.283 PR-AUC**, while HGB gains about **+0.217 ROC-AUC and +0.317 PR-AUC**. Because the model settings are unchanged within each family, these jumps are attributable to the post-call feature set rather than tuning.

The **tuned pre-call-to-tuned +duration comparison** shows that the duration advantage remains even after each feature regime selects its own hyperparameters: LR gains about **+0.224 ROC-AUC and +0.280 PR-AUC**; HGB gains about **+0.209 ROC-AUC and +0.277 PR-AUC**.

Tuning itself is not the main source of performance. LR tuning is essentially neutral/slightly negative on the holdout. HGB tuning gives a small pre-call PR-AUC improvement (**0.2683 → 0.2728**), while the +duration HGB winner selected by validation PR-AUC shifts strongly toward recall because it uses balanced class weights; on the holdout this lowers ROC-AUC/PR-AUC relative to the untuned +duration HGB.

Therefore the correct conclusion is: **the large apparent performance increase comes from post-call duration, not from tuning.** `duration` remains diagnostic only because it is unavailable when the bank must decide whom to call.
